# 02 — Latest Yamada optimization benchmark and equivalence checks

Run this notebook from the `perf/yamada-max-optimization` checkout. It benchmarks the current branch source rather than a site-packages installation.

It executes the repository's benchmark/oracle scripts and rejects a benchmark if the optimized result differs from the retained reference implementation. It covers the original SymPy/NetworkX evaluator, the intermediate exact-Laurent evaluator, the compact integer-multigraph evaluator, end-to-end 3D→projection→PD→Yamada scaling, dispatch behavior, and projection hot spots.

Parsed results are written into `User_guide/benchmarks/results_latest/`; figures go to `figures_latest/`.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

branch = subprocess.check_output(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=ROOT, text=True
).strip()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True
).strip()

print("ROOT   =", ROOT)
print("branch =", branch)
print("commit =", commit)

if branch != "perf/yamada-max-optimization":
    raise RuntimeError(
        "This notebook is intended for perf/yamada-max-optimization; "
        f"current branch is {branch!r}."
    )

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
print("knotted_graph loaded from:", kg_path)
if SRC not in kg_path.parents:
    raise RuntimeError(
        "A stale installed knotted_graph was imported instead of this checkout."
    )

BENCH = ROOT / "User_guide" / "benchmarks"
RESULTS = BENCH / "results_latest"
FIGURES = BENCH / "figures_latest"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

In [ ]:
import csv
import matplotlib.pyplot as plt

DEV = ROOT / "dev"

def run_summary(script, *args, timeout=1800):
    cmd = [sys.executable, str(DEV / script), *map(str, args)]
    env = dict(os.environ)
    env["PYTHONPATH"] = str(SRC) + os.pathsep + env.get("PYTHONPATH", "")
    p = subprocess.run(cmd, cwd=ROOT, env=env, text=True, capture_output=True, timeout=timeout)
    print(p.stdout)
    if p.returncode:
        print(p.stderr)
        raise RuntimeError(f"{script} failed with return code {p.returncode}")
    summary = None
    for line in p.stdout.splitlines():
        if line.startswith("SUMMARY="):
            summary = json.loads(line[len("SUMMARY="):])
    return summary, p.stdout

def save_rows(name, rows):
    if rows is None:
        return
    path = RESULTS / name
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print("saved", path.relative_to(ROOT))

## A. Crossing-free kernel: original → exact Laurent → compact

In [ ]:
kernel_rows, _ = run_summary("benchmark_yamada_kernel.py")
assert kernel_rows, "No kernel results returned."
save_rows("02_kernel_speedups.csv", kernel_rows)
for r in kernel_rows:
    print(
        f"{r['case']:12s} E={r['E']:2d} "
        f"direct={r['compact_direct_speedup']:8.1f}x "
        f"negami={r['compact_negami_speedup']:8.1f}x"
    )

In [ ]:
plt.figure(figsize=(8.5,5.3))
q=sorted(kernel_rows,key=lambda r:(r["E"],r["case"]))
plt.plot([r["E"] for r in q],[r["compact_direct_speedup"] for r in q],marker="o",label="Direct")
plt.plot([r["E"] for r in q],[r["compact_negami_speedup"] for r in q],marker="o",label="Negami")
plt.yscale("log")
plt.xlabel("Edges E")
plt.ylabel("Original / optimized runtime")
plt.title("Exact compact Yamada kernel speedup")
plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES/"02_kernel_speedup.pdf",bbox_inches="tight")
plt.savefig(FIGURES/"02_kernel_speedup.png",dpi=300,bbox_inches="tight")
plt.show()

## B. End-to-end crossing scaling on the optimized branch

In [ ]:
e2e_rows, _ = run_summary("benchmark_yamada_end_to_end.py")
assert e2e_rows
save_rows("02_end_to_end_current.csv", e2e_rows)

In [ ]:
plt.figure(figsize=(8.5,5.3))
q=sorted(e2e_rows,key=lambda r:r["crossings"])
plt.plot([r["crossings"] for r in q],[r["runtime_s"] for r in q],marker="o")
plt.yscale("log")
plt.xlabel("Projected crossings c")
plt.ylabel("3D → projection → PD → Yamada runtime (s)")
plt.title("Optimized end-to-end crossing scaling")
plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(FIGURES/"02_end_to_end_crossings.pdf",bbox_inches="tight")
plt.savefig(FIGURES/"02_end_to_end_crossings.png",dpi=300,bbox_inches="tight")
plt.show()

## C. Dispatch and projection hot spots

In [ ]:
for script in [
    "benchmark_yamada_dispatch.py",
    "benchmark_projection_index.py",
    "benchmark_crossing_bulk_query.py",
    "benchmark_crossing_vectorized.py",
]:
    print("\n###", script)
    summary, text = run_summary(script)
    if summary:
        save_rows("02_" + script.replace(".py",".csv"), summary)

## Acceptance criterion

Every benchmark script contains correctness assertions. A timing result is not accepted if the optimized polynomial or geometric crossing assignment differs from the retained reference path.